# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding A — ML Appendix: Feature Importance for Health Score (p.27)

**Finding:** Random Forest ranks Average Position (43%) and Impressions (32%) as the top
predictors of Health Score.

**Where the label comes from:** Health Score is a composite the paper defines itself:
`impressions(30) + position(30) + ctr(20) + scroll(20)`. It's not an independent outcome —
it's a weighted formula built from some of the same columns fed into the model as features.

**My methodology question:** Two of the top three "predictors" are literally components the
target is built from. Does this importance ranking tell us anything new about what drives
Health Score, or does it mostly confirm the formula's own weights back to us? The paper
discloses this directly ("importance is descriptive rather than causal") — but a reader
skimming just the bar chart could easily read it as "optimize position and impressions" as if
it were a fresh discovery, rather than a restatement of the scoring formula.

**Disclosure vs. over-conclusion risk:** Low risk in the paper itself (it's disclosed in the
chart-read note). The risk sits in how a reader *acts* on the chart alone without reading the caveat.

---

### Finding B — Finding #4: Freshness Multiplier, the 361+ bucket (p.9)

**Finding:** Pages untouched for 361+ days show a 283:1 growth-to-decline ratio.

**Where the label comes from:** Growth/decline is the same 30d-vs-prev-30d trend bucket used
throughout the paper (`trend_direction`), grouped by `days_since_last_update`.

**My methodology question:** The paper states this bucket has only 1 declining page against
283 growing ones. With n=1 on the "decline" side, does the validation design (or even basic
descriptive-stats hygiene) carry a 283:1 ratio as a usable number, or is this closer to noise
that happens to look dramatic? The paper's own `auditing-signals`-style floor (n≥50 per bucket)
would flag this bucket as too small to headline.

**Disclosure vs. over-conclusion risk:** The paper is careful here too — it explicitly says
"too small and too unstable to treat as a headline multiplier" and recommends the 31-90 day
window (7.88:1, much larger n) as the real signal instead. Good practice: they showed the
number but didn't let it become the takeaway.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

# Load your Week-5 feature vector (adjust path if yours differs)
df = pd.read_csv("../../data/processed/refresh_feature_vector.csv")

from scripts_or_your_utils import build_feature_matrix  # or paste the function from ml_utils.py
X, feature_cols = build_feature_matrix(df)
y = df["is_declining_label"].astype(int)

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(y_true)[order[:k]].mean()

# BEFORE: naive random split (dishonest — client rows can leak across train/test)
X_tr, X_te, y_tr, y_te, idx_tr, idx_te = train_test_split(
    X, y, df.index, test_size=0.2, random_state=42, stratify=y
)
model_random = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=25,
    class_weight="balanced_subsample", random_state=42, n_jobs=-1
).fit(X_tr, y_tr)
probs_random = model_random.predict_proba(X_te)[:, 1]
p50_random = precision_at_k(y_te.values, probs_random, 50)

# AFTER: grouped by client (honest — no client appears in both sides)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))
model_grouped = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=25,
    class_weight="balanced_subsample", random_state=42, n_jobs=-1
).fit(X.iloc[train_idx], y.iloc[train_idx])
probs_grouped = model_grouped.predict_proba(X.iloc[test_idx])[:, 1]
p50_grouped = precision_at_k(y.iloc[test_idx].values, probs_grouped, 50)

print(f"Random split  Precision@50: {p50_random:.3f}")
print(f"Client-grouped Precision@50: {p50_grouped:.3f}")
print(f"Gap: {p50_random - p50_grouped:+.3f}")

**Interpretation:** [fill in after running] — e.g. "the random split overstated Precision@50
by X points, consistent with the model partly memorizing client-specific patterns rather than
generalizing to unseen clients."

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# Prove the leak, don't just claim it
X_leaky = X.copy()
X_leaky["trend_pct"] = df["trend_pct"].values

train_idx, test_idx = next(gss.split(X_leaky, y, groups=df["client_id"]))
leaky_model = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=25,
    class_weight="balanced_subsample", random_state=42, n_jobs=-1
).fit(X_leaky.iloc[train_idx], y.iloc[train_idx])
leaky_probs = leaky_model.predict_proba(X_leaky.iloc[test_idx])[:, 1]
p50_leaky = precision_at_k(y.iloc[test_idx].values, leaky_probs, 50)

print(f"Clean model  Precision@50: {p50_grouped:.3f}")
print(f"Leaky model  Precision@50: {p50_leaky:.3f}  <- trend_pct added back in")

**Other features checked (available before the decision point?):**
- `days_since_last_update` — snapshot value, known before prediction. Safe.
- `avg_position`, `ctr` — 90-day trailing aggregates, known before prediction. Safe.
- `trend_direction`, `trend_pct` — derived from the same 30d comparison the label uses. Excluded.
- `content_id`/`client_id` — used for grouping only, never as features. Confirmed not in `feature_cols`.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Before:** "The model predicts which pages will decline."

**After:** "On this anonymized 30K-row slice, under a client-grouped holdout split, the model
ranks pages by decline-risk at Precision@50 of [X] — roughly [X*50] of the top 50 flagged pages
were labeled declining, versus [baseline] for the hand-written rule. This is evidence the model
found a real, if imperfect, ranking signal on held-out clients — not proof it forecasts future
traffic, since the label itself is a same-window bucket (`trend_direction`), not a validated
future outcome."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.